---
#**Instalação**
---

In [ ]:
!pip install pyarrow -q

In [ ]:
import zipfile
import io
import requests
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from tqdm.auto import tqdm
import json
from sklearn.preprocessing import MinMaxScaler
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from IPython.display import display
import numpy as np

---
#**Configurações**
---

In [ ]:
TABLE_SCR  = "scr_data"
TABLE_PIX  = "pix_transacoes"
TABLES_IBGE = ["taxa_escolarizacao", "censo_demografico", "populacao_por_idade"]

PASTA = Path("/content/dados")
PASTA.mkdir(exist_ok=True)

#Rodar com o localtunnel
URL_API_LOCAL = "https://safer-survivor-kerry-doctrine.trycloudflare.com"
SESSION = requests.Session()
SESSION.headers.update({"bypass-tunnel-reminder": "true"})

CHUNK_SIZE = 10000
MAX_GET_RETRIES = 5

#adicionado coluna modalidade
USECOLS_SCR = ["data_base", "uf", "porte", "cliente", "modalidade",
    "carteira_inadimplencia", "carteira_vencida", "carteira_ativa", "vencido_15_a_60_dias", "vencido_61_a_90_dias", "vencido_acima_de_90_dias"]

ano_anterior = datetime.now().year - 1
URLS_SCR = [f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip" for ano in range(2021, ano_anterior+1)]

SCR_LIMITE_DADOS = 100000

NIVEIS_SALARIO = {
    # Salário Pessoa Física (PF)
    "Acima de 20 salários mínimos": "A",
    "Mais de 10 a 20 salários mínimos": "A",
    "Mais de 5 a 10 salários mínimos": "B",
    "Mais de 3 a 5 salários mínimos": "C",
    "Mais de 2 a 3 salários mínimos": "D",
    "Mais de 1 a 2 salários mínimos": "D",
    "Até 1 salário mínimo": "E",
    "Sem rendimento": "E",
    "a": "A",

    # Salário Pessoa Jurídica (PJ)
    "Grande": "A",
    "Médio": "B",
    "Pequeno": "C",
    "Micro": "D",
}

PIX_LIMITE_DADOS = 35000
URL_PIX_API = f"https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?@DataBase='<ANO_MES>'&$top={PIX_LIMITE_DADOS}&$format=json&$select=AnoMes,Estado_Ibge,VL_PagadorPF,QT_PagadorPF,VL_PagadorPJ,QT_PagadorPJ,QT_PES_PagadorPF,QT_PES_PagadorPJ"

LINKS_IBGE = {
    "taxa_escolarizacao": {
      "url": "https://apisidra.ibge.gov.br/values/t/7138/n3/all/v/10276/p/last%203/c2/6794/c58/100052,108866/d/v10276%201?formato=json",
      "use_cols": ["V", "D1N", "D3N"]
    },
    "censo_demografico": {
        "url": "https://apisidra.ibge.gov.br/values/t/4709/n1/all/n2/all/n3/all/v/all/p/all/d/v10605%202/l/p,v,t?formato=json",
        "use_cols": ["V" ,"D1N", "D2N", "D3N"]
    },
    "populacao_por_idade": {
        "url": "https://apisidra.ibge.gov.br/values/t/9514/n3/all/v/allxp/p/all/c2/6794/c287/6653,49108,49109,60040,60041,93070,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93095,93096,93097/c286/113635?formato=json",
        "use_cols": ["V", "D1N", "D3N", "D5N"]
    }
}

map_idade_media = {
    "0 a 4 anos": 2.0,
    "5 a 9 anos": 7.0,
    "10 a 14 anos": 12.0,
    "15 a 19 anos": 17.0,
    "20 a 24 anos": 22.0,
    "25 a 29 anos": 27.0,
    "30 a 34 anos": 32.0,
    "35 a 39 anos": 37.0,
    "40 a 44 anos": 42.0,
    "45 a 49 anos": 47.0,
    "50 a 54 anos": 52.0,
    "60 a 64 anos": 62.0,
    "65 a 69 anos": 67.0,
    "70 a 74 anos": 72.0,
    "80 a 84 anos": 82.0,
    "85 a 89 anos": 87.0,
    "90 a 94 anos": 92.0,
    "95 a 99 anos": 97.0,
    "100 anos ou mais": 100.0
}

---
#**Funções de Leitura**
---

In [ ]:
def baixar_com_retry(url, SESSION, TIMEOUT):  #Leitura geral de arquivos
  primeria_tentativa = True
  for i in range(1, MAX_GET_RETRIES + 1): #Faz get com limite de tentativas
    try:
      resposta = SESSION.get(url, timeout=TIMEOUT) #Tenta em um tempo curto
      resposta.raise_for_status()
      resposta.encoding = "utf-8-sig"
      return resposta
    except Exception as e:
      if i < MAX_GET_RETRIES:
        pass
      else:
        print(f"Todas as {MAX_GET_RETRIES} tentativas falharam.")
        raise e
        return

In [ ]:
def ler_json(url, SESSION, timeout = 5, chave = None, funcao_tratamento = None, filtro_colunas = None):
  resposta = baixar_com_retry(url, SESSION, timeout)
  if resposta is None:
    return

  json = resposta.json()

  if(chave and chave in json):
    df = pd.DataFrame(json[chave])
  else:
    df = pd.DataFrame(json)

  if filtro_colunas:
    df = df[filtro_colunas]
  if funcao_tratamento:
    return funcao_tratamento(df)

  return df

---
#**Função para enviar dados**
---

In [ ]:
def enviar(rota, dados):
    url = URL_API_LOCAL + rota
    for item in dados:
        try:
            resposta = SESSION.post(url, json=item, timeout=15)
            if resposta.status_code != 201:
                print(f"Erro {resposta.status_code} em {rota}: {resposta.text}")
                break
        except Exception as e:
            print(f"Erro: {e}")
            break

---
#**Leitura dados SCR**
---

In [ ]:
def processar_url_scr(args):
    url, SESSION = args

    tqdm.write(f"> Baixando: {url.split('/')[-1]}...")

    resposta = baixar_com_retry(url, SESSION, 30)
    if resposta is None:
        return []

    dfs = []
    with zipfile.ZipFile(io.BytesIO(resposta.content)) as zf:
        csvs = [nome for nome in zf.namelist() if nome.endswith(".csv")]

        for nome_arquivo in csvs:
            with zf.open(nome_arquivo) as f:
                leitor = pd.read_csv(
                    f, sep=';', encoding="utf-8-sig", on_bad_lines="skip",
                    usecols=USECOLS_SCR, chunksize=50000
                )

                for i, chunk in enumerate(leitor):
                    df_tratado = tratamento_scr(chunk)

                    if not df_tratado.empty:
                        dfs.append(df_tratado)

    return dfs

def ler_scr(SESSION):
    ano_atual = datetime.now().year
    URLS_SCR_FILTRADAS = [f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip" for ano in range(2021, ano_atual + 1)]

    args = [(url, SESSION) for url in URLS_SCR_FILTRADAS]
    with ThreadPoolExecutor() as executor:
        resultados = list(tqdm(executor.map(processar_url_scr, args), total=len(URLS_SCR_FILTRADAS), desc="Baixando arquivos SRC"))

    dfs = [df for lista in resultados for df in lista]
    if not dfs:
        print("Nenhum dado do SCR encontrado.")
        return

    df_final = pd.concat(dfs, ignore_index=True)
    print(f"Lido SCR: {df_final.shape[0]} linhas, {df_final.shape[1]} colunas")
    salvar_parquet(df_final, TABLE_SCR)

---
#**Leitura dados Pix**
---

In [ ]:
def gerar_anos_meses_filtrado():
    data_atual = datetime.now()
    anos_meses = pd.date_range(start="2020-11-01", end=data_atual, freq="MS")
    lista_completa = anos_meses[:-1].strftime("%Y%m").tolist()

    return lista_completa

In [ ]:
def processar_url_pix(args):
    url, SESSION = args
    return ler_json(url=url, SESSION=SESSION, timeout=10, chave="value", funcao_tratamento=tratamento_pix)

def ler_pix(SESSION):
    anos_meses = gerar_anos_meses_filtrado()

    URLS_PIX = [URL_PIX_API.replace("<ANO_MES>", ano_mes) for ano_mes in anos_meses]

    args = [(url, SESSION) for url in URLS_PIX]
    with ThreadPoolExecutor() as executor:
        resultado = list(tqdm(executor.map(processar_url_pix, args), total=len(URLS_PIX), desc="Baixando arquivos pix..."))

    if not any(r is not None and not r.empty for r in resultado):
        print("Nenhum dado do PIX encontrado.")
        return

    df = pd.concat([r for r in resultado if r is not None and not r.empty], ignore_index=True)
    print(f"Lido PIX: {df.shape[0]} linhas, {df.shape[1]} colunas")
    salvar_parquet(df, TABLE_PIX)

---
#**Leitura dados IBGE**
---

In [ ]:
def ler_urls_ibge(SESSION):
    funcoes_tratamento_ibge = [tratamento_taxa_escolarizacao, tratamento_censo_demografico, tratamento_populacao_por_idade]
    tratamentos = {k: v for k, v in zip(TABLES_IBGE, funcoes_tratamento_ibge)}

    for table_name, item in LINKS_IBGE.items():
        print(f"Baixando {table_name}...")
        df = ler_json(item["url"], SESSION=SESSION, timeout=3, filtro_colunas=item["use_cols"], funcao_tratamento=tratamentos[table_name])

        if df is not None and not df.empty:
            salvar_parquet(df, table_name)
        else:
            print(f"Nenhum dado encontrado para {table_name}.")

---
#**Tratamento**
---

In [ ]:
#Funções gerais de tratamento de dados

def limpeza_string(df, colunas=None):
    if colunas is None:
        colunas = df.select_dtypes(include="object").columns.tolist()
    else:
        colunas = [col for col in colunas if col in df.columns and df[col].dtype == "object"]

    if not colunas:
        return df

    for col in colunas:
        df[col] = df[col].str.strip()
    return df

def converter_numerico(df, *colunas):
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def limpar_nulos(df):
    numericas = df.select_dtypes(include="number").columns
    df[numericas] = df[numericas].fillna(df[numericas].median())

    categoricas = df.select_dtypes(include="object").columns
    df[categoricas] = df[categoricas].fillna("NAO_INFORMADO")
    return df

def formatar_data(df, colunas, formato="%Y-%m-%d"):
    if not colunas: return df
    if isinstance(colunas, str):
        colunas = [colunas]
    for col in colunas:
        df[col] = pd.to_datetime(df[col], format=formato).dt.strftime("%Y%m")
    return df

def formatar_numero(df, colunas):
    if isinstance(colunas, str):
        colunas = [colunas]
    for col in colunas:
        df[col] = pd.to_numeric(df[col].str.replace(",", "."), errors="coerce")
    return df

def converter_nivel_salario(df, coluna):
  df[coluna] = df[coluna].map(NIVEIS_SALARIO)
  return df

def converter_uf(df, coluna):
  estados_brasil = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF", "Espírito Santo": "ES",
    "Goiás": "GO", "Maranhão": "MA", "Mato Grosso": "MT", "Mato Grosso do Sul": "MS",
    "Minas Gerais": "MG", "Pará": "PA", "Paraíba": "PB", "Paraná": "PR",
    "Pernambuco": "PE", "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN",
    "Rio Grande do Sul": "RS", "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC",
    "São Paulo": "SP", "Sergipe": "SE", "Tocantins": "TO",
    11: "RO", 12: "AC", 13: "AM", 14: "RR",
    15: "PA", 16: "AP", 17: "TO", 21: "MA",
    22: "PI", 23: "CE", 24: "RN", 25: "PB",
    26: "PE", 27: "AL", 28: "SE", 29: "BA",
    31: "MG", 32: "ES", 33: "RJ", 35: "SP",
    41: "PR", 42: "SC", 43: "RS", 50: "MS",
    51: "MT", 52: "GO", 53: "DF"
  }

  df = df.rename(columns = {
      coluna: "uf"
  })
  df["uf"] = df["uf"].map(estados_brasil)

  return df

def converter_uf_para_macro_regiao(df, coluna):
  mapeamento_regioes = {
      'AC': 'Norte', 'AM': 'Norte', 'AP': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
      'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
      'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
      'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste',
      'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
      'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
  }

  df['regiao'] = df[coluna].map(mapeamento_regioes)

  return df

def converter_porcentagem(df, coluna):
  df[coluna] = pd.to_numeric(df[coluna], errors='coerce') / 100
  return df

In [ ]:
def tratamento_scr(df):
    # 1. Limpeza básica de textos, datas, regiões e portes salariais
    df = limpar_nulos(df)
    df = limpeza_string(df, ["data_base", "uf", "porte"])
    df = formatar_data(df, ["data_base"])
    df = converter_uf_para_macro_regiao(df, "uf")
    df = converter_nivel_salario(df, "porte")

    # 2. Identifica dinamicamente todas as colunas de carteira e atrasos (vencido_)
    colunas_financeiras = [
        col for col in df.columns
        if col.startswith("carteira_") or col.startswith("vencido_")
    ]

    # 3. Formata os números (troca vírgula por ponto e converte para numérico)
    df = formatar_numero(df, colunas_financeiras)

    # 4. TRATAMENTO DE NULOS: ausência de registro = R$ 0.00 de atraso/carteira
    df[colunas_financeiras] = df[colunas_financeiras].fillna(0.0)
    return df

In [ ]:
def tratamento_pix(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["AnoMes"])
  df = converter_numerico(df, "VL_PagadorPF", "QT_PagadorPF", "VL_PagadorPJ", "QT_PagadorPJ", "QT_PES_PagadorPF", "QT_PES_PagadorPJ")
  df = converter_uf(df, "Estado_Ibge")
  df = converter_uf_para_macro_regiao(df, "uf")
  return df

In [ ]:
def tratamento_taxa_escolarizacao(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["D1N", "D3N"])
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = converter_porcentagem(df, "V")
  df = df.rename(columns={"D3N": "ano", "V": "valor"})
  return df

def tratamento_censo_demografico(df):
  df = limpar_nulos(df)
  df["V"] = df["V"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
  df = converter_numerico(df, "V")
  df = limpeza_string(df, ["D1N", "D3N"])
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = df.rename(columns={"D3N": "ano", "V": "valor"})
  return df

def tratamento_populacao_por_idade(df):
  df = limpar_nulos(df)
  df = limpeza_string(df, ["D1N", "D3N", "D5N"])
  df = converter_numerico(df, "V")
  df = converter_uf(df, "D1N")
  df = converter_uf_para_macro_regiao(df, "uf")
  df = df.rename(columns={"D3N": "ano", "V": "valor", "D5N": "idade"})
  df["idade"] = df["idade"].map(map_idade_media)
  return df

---
#**Perpetuação**
---

In [ ]:
def salvar_parquet(df, nome):
  caminho = PASTA / f"{nome}.parquet"
  df.to_parquet(caminho, index=False)
  print(f" --Salvo em: {caminho}")

# **Refatoração**

In [ ]:
def validar_consistencia_saldos(df):
    """Função para checar se a soma dos saldos por modalidade é igual à carteira ativa total."""
    # 1. Filtra registros de carteira ativa
    df_ativa = df[df["metrica"] == "carteira_ativa"]

    # 2. Agrupa por modalidade e calcula a soma total
    soma_modalidades = (
        df_ativa.groupby(["ano_mes", "uf", "modalidade"])["valor"]
        .sum()
        .groupby(["ano_mes", "uf"])
        .sum()
        .reset_index(name="soma_modalidades")
    )

    # 3. Obtém o total da carteira ativa informado para o período/UF
    total_carteira = (
        df_ativa.groupby(["ano_mes", "uf"])["valor"]
        .sum()
        .reset_index(name="total_carteira_ativa")
    )

    # 4. Cruza e valida a diferença
    checagem = pd.merge(soma_modalidades, total_carteira, on=["ano_mes", "uf"])
    checagem["diferenca"] = (
        checagem["soma_modalidades"] - checagem["total_carteira_ativa"]
    ).round(2)

    inconsistencias = checagem[checagem["diferenca"] != 0]

    if not inconsistencias.empty:
        print(
            f"[VALIÇÃO] Encontradas {len(inconsistencias)} inconsistências entre modalidades e carteira ativa!"
        )
    else:
        print(
            "[VALIÇÃO] Saldos entre modalidades e carteira ativa estão 100% consistentes!"
        )


def padronizar_df_scr(df_scr):
    df_scr = df_scr.rename(columns={"data_base": "ano_mes", "porte": "classe"})

    df_scr["tipo"] = df_scr["cliente"].map({"PF": "pf"}).fillna("pj")

    # Inclui 'modalidade' id_vars para preservar o detalhamento das modalidades
    df_scr = df_scr.melt(
        id_vars=["ano_mes", "regiao", "uf", "tipo", "classe", "modalidade"],
        value_vars=[
            "carteira_inadimplencia",
            "carteira_vencida",
            "carteira_ativa",
            "vencido_acima_de_90_dias",
        ],
        var_name="metrica",
        value_name="valor",
    )

    df_scr["origem"] = "scr"

    validar_consistencia_saldos(df_scr)

    for col in ["tipo", "classe", "origem", "metrica", "uf", "modalidade"]:
        if col in df_scr.columns:
            df_scr[col] = df_scr[col].astype("category")

    df_scr = (
        df_scr.groupby(
            [
                "ano_mes",
                "regiao",
                "uf",
                "tipo",
                "classe",
                "modalidade",
                "metrica",
                "origem",
            ],
            observed=True,
        )["valor"]
        .sum()
        .reset_index()
    )

    return df_scr[[
        "ano_mes",
        "regiao",
        "uf",
        "tipo",
        "classe",
        "metrica",
        "origem",
        "valor",
    ]]

In [ ]:
import numpy as np
import pandas as pd


def padronizar_df_pix(df_pix):
    df_pix = df_pix.rename(columns={"AnoMes": "ano_mes"})

    df_pix = df_pix.melt(
        id_vars=["ano_mes", "regiao", "uf"],
        value_vars=[
            "VL_PagadorPF",
            "QT_PagadorPF",
            "VL_PagadorPJ",
            "QT_PagadorPJ",
            "QT_PES_PagadorPF",
            "QT_PES_PagadorPJ",
        ],
        var_name="metrica_raw",
        value_name="valor",
    )

    df_pix["tipo"] = (
        df_pix["metrica_raw"].str.endswith("PF").map({True: "pf", False: "pj"})
    )

    df_pix["metrica"] = np.select(
        [
            df_pix["metrica_raw"].str.contains("VL"),
            df_pix["metrica_raw"].str.contains("QT_PES"),
            df_pix["metrica_raw"].str.contains("QT"),
        ],
        ["vl_pagador", "qt_pes_pagador", "qt_pagador"],
        default="outros",
    )

    df_pix["origem"] = "pix"

    df_pix = (
        df_pix.groupby(
            ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
            observed=True,
        )["valor"]
        .sum()
        .reset_index()
    )

    # Compatibilidade de colunas com o schema do SCR
    df_pix["classe"] = pd.NA
    df_pix["modalidade"] = "pix"
    df_pix["categoria_credito"] = "nao_aplica"

    for col in [
        "tipo",
        "origem",
        "metrica",
        "uf",
        "regiao",
        "classe",
        "modalidade",
        "categoria_credito",
    ]:
        if col in df_pix.columns:
            df_pix[col] = df_pix[col].astype("category")

    return df_pix[[
        "ano_mes",
        "regiao",
        "uf",
        "tipo",
        "classe",
        "modalidade",
        "categoria_credito",
        "metrica",
        "origem",
        "valor",
    ]]

In [ ]:
import pandas as pd

# Execução do pipeline de padronização
df_scr_pronto = padronizar_df_scr(df_scr)
df_pix_pronto = padronizar_df_pix(df_pix)

# Unificação das duas bases em um único DataFrame consolidado
df_consolidado = pd.concat(
    [df_scr_pronto, df_pix_pronto], ignore_index=True
)

# Exibição do resumo das categorias de crédito por modalidade e origem
print("--- Resumo por Origem e Categoria de Crédito ---")
print(
    df_consolidado.groupby(
        ["origem", "modalidade", "categoria_credito"], observed=True
    )["valor"].sum()
)

NameError: name 'df_scr' is not defined

---
#**Pré Processamento**
---

In [ ]:
def processar():
    print(f"--- Iniciando Extração de Dados ---")

    SESSION = requests.Session()

    try:
        ler_scr(SESSION)
    except Exception as e:
        print(f"Erro ao ler SCR: {e}")

    try:
        ler_pix(SESSION)
    except Exception as e:
        print(f"Erro ao ler Pix: {e}")

    try:
        ler_urls_ibge(SESSION)
    except Exception as e:
        print(f"Erro ao ler IBGE: {e}")

---
#**Scoring**
---

In [ ]:
def extract_data(path:str):
    df = pd.read_parquet(path)
    return df

Leitura Parquets Pix e Scr
---

In [ ]:
def padronizar_df_scr(df_scr):
    df_scr = df_scr.rename(columns={"data_base": "ano_mes", "porte": "classe"})

    df_scr["tipo"] = df_scr["cliente"].map({"PF": "pf"}).fillna("pj")

    df_scr = df_scr.melt(
        id_vars=["ano_mes", "regiao", "uf", "tipo", "classe"],
        value_vars=["carteira_inadimplencia", "carteira_vencida", "carteira_ativa", "vencido_15_a_60_dias", "vencido_61_a_90_dias", "vencido_acima_de_90_dias"],
        var_name="metrica",
        value_name="valor"
    )

    df_scr["origem"] = "scr"

    for col in ["tipo", "classe", "origem", "metrica", "uf"]:
        if col in df_scr.columns:
            df_scr[col] = df_scr[col].astype("category")

    df_scr = df_scr.groupby(
      ["ano_mes", "regiao", "uf", "tipo", "classe", "metrica", "origem"],
      observed=True
      )["valor"].sum().reset_index()

    return df_scr[["ano_mes", "regiao", "uf", "tipo", "classe", "metrica", "origem", "valor"]]

In [ ]:
def padronizar_df_pix(df_pix):
    df_pix = df_pix.rename(columns={"AnoMes": "ano_mes"})

    df_pix = df_pix.melt(
        id_vars=["ano_mes", "regiao", "uf"],
        value_vars=["VL_PagadorPF", "QT_PagadorPF", "VL_PagadorPJ", "QT_PagadorPJ", "QT_PES_PagadorPF", "QT_PES_PagadorPJ"],
        var_name="metrica_raw",
        value_name="valor"
    )

    df_pix["tipo"] = df_pix["metrica_raw"].str.endswith("PF").map({True: "pf", False: "pj"})

    df_pix["metrica"] = np.select(
        [df_pix["metrica_raw"].str.contains("VL"),
         df_pix["metrica_raw"].str.contains("QT_PES"),
         df_pix["metrica_raw"].str.contains("QT")],
        ["vl_pagador", "qt_pes_pagador", "qt_pagador"],
        default="outros"
    )

    df_pix["origem"] = "pix"

    df_pix = df_pix.groupby(
        ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
        observed=True
    )["valor"].sum().reset_index()

    df_pix["classe"] = pd.NA  # adiciona depois do groupby

    for col in ["tipo", "origem", "metrica", "uf", "regiao"]:
        df_pix[col] = df_pix[col].astype("category")

    return df_pix[["ano_mes", "regiao", "uf", "tipo", "classe", "metrica", "origem", "valor"]]

In [ ]:
def construir_df_scr_pix():
  chunks_pix = []
  chunks_scr = []

  #Padroniza o parquet pix por chunk
  tabela_pix = pq.ParquetFile('/content/dados/pix_transacoes.parquet')

  for batch in tabela_pix.iter_batches(batch_size=100000):
    chunks_pix.append(padronizar_df_pix(batch.to_pandas()))
  df_pix = pd.concat(chunks_pix, ignore_index=True)

  df_pix = df_pix.groupby(
      ["ano_mes", "regiao", "uf", "tipo", "metrica", "origem"],
      observed=True
  )["valor"].sum().reset_index()
  df_pix["classe"] = pd.NA

  #Padroniza o parquet scr por chunk
  tabela_scr = pq.ParquetFile('/content/dados/scr_data.parquet')

  for batch in tabela_scr.iter_batches(batch_size=100000):
    chunks_scr.append(padronizar_df_scr(batch.to_pandas()))
  df_scr = pd.concat(chunks_scr, ignore_index=True)

  df_scr = df_scr.groupby(
      ["ano_mes", "regiao", "uf", "tipo", "classe", "metrica", "origem"],
      observed=True
  )["valor"].sum().reset_index()

  df_pix["ano_mes"] = df_pix["ano_mes"].astype(str)
  df_scr["ano_mes"] = df_scr["ano_mes"].astype(str)

  #une os dois parquets em um individual
  df_scr_pix = pd.concat([df_scr, df_pix], ignore_index=True)

  salvar_parquet(df_scr_pix, "scr_pix")

Leitura Parquets IBGE
---

In [ ]:
def calculo_idade():
  #---populacao por idade
  df_idade = pd.read_parquet('/content/dados/populacao_por_idade.parquet')

  df_idade["idade_ponderada"] = df_idade["idade"] * df_idade["valor"]

  df_idade = (
      df_idade.groupby(["ano", "regiao", "uf"])
      .agg(
          soma_idade=("idade_ponderada", "sum"),
          soma_pop=("valor", "sum")
      )
      .reset_index()
  )

  df_idade["idade_media"] = df_idade["soma_idade"] / df_idade["soma_pop"]

  df_idade["idade_max"] = df_idade.groupby("ano")["idade_media"].transform("max")

  df_idade["bonusDemografico"] = df_idade["idade_max"] - df_idade["idade_media"]

  return df_idade[["ano", "regiao", "uf", "bonusDemografico"]]

In [ ]:
def construir_df_ibge():
  #---escolarizacao
  df_escolarizacao = extract_data('/content/dados/taxa_escolarizacao.parquet')

  df_escolarizacao = df_escolarizacao.groupby(  #realiza a média de valores com mesmo ano, regiao e uf
      ["ano", "regiao", "uf"],
      observed=True)["valor"].mean().reset_index()
  df_escolarizacao = df_escolarizacao.rename(columns={"valor": "taxa_escolarizacao"})

  #---censo demografico
  df_censo_demografico = extract_data('/content/dados/censo_demografico.parquet')

  #separa colunas por metrica
  df_censo_demografico = df_censo_demografico.pivot_table(
      index=["ano", "regiao", "uf"],
      columns="D2N",
      values="valor").reset_index()

  df_censo_demografico = df_censo_demografico.rename(columns={
      "População residente": "populacao_residente",
      "Taxa de crescimento geométrico": "taxa_crescimento",
      "Variação absoluta da população residente 2010 compatibilizada": "variacao_populacao"
  })

  df_idade = calculo_idade()

  #arruma porcentagem
  df_censo_demografico["taxa_crescimento"] = df_censo_demografico["taxa_crescimento"]/100

  df_censo_demografico.columns.name = None

  df_ibge = df_escolarizacao.merge(
      df_censo_demografico,
      on=["ano", "regiao", "uf"],
      how="left").merge(
      df_idade,
      on=["ano", "regiao", "uf"],
      how="left")

  df_ibge = (
      df_ibge.sort_values(["uf", "regiao", "ano"])
      .ffill()
      .reset_index()
  )

  salvar_parquet(df_ibge, "ibge")

In [ ]:
def normalizacao(df, coluna):
  df[coluna + "_max"] = df.groupby("ano_mes")[coluna].transform("max")
  df[coluna + "_min"] = df.groupby("ano_mes")[coluna].transform("min")
  df[coluna] = (1 + (df[coluna] - df[coluna + "_min"]) / (df[coluna + "_max"] - df[coluna + "_min"]) * 4).round(2)
  df = df.drop(columns=[coluna + "_max", coluna + "_min"])

  return df

---
#**Eixo I – Risco de Crédito**
---

In [ ]:
filtro = ["ano_mes", "regiao", "uf"]

def agrupar_metrica(df, metrica, filtro_extra=None):
    mask = df["metrica"] == metrica
    if filtro_extra is not None:
        mask &= filtro_extra
    return df[mask].groupby(filtro)["valor"].sum()

In [ ]:
# --- Mudança feita dia 24-09 (Miguel) ---

def calcular_eixo_i(df_scr_pix_base, df_ibge_base):
    df_scr_pix = df_scr_pix_base.copy()
    df_ibge = df_ibge_base.copy()

    # --- Extrai ano e merge com IBGE ---
    df_scr_pix["ano"] = df_scr_pix["ano_mes"].astype(str).str[:4]

    df_calculos = df_scr_pix.merge(
        df_ibge[["ano", "regiao", "uf", "taxa_escolarizacao"]],
        on=["ano", "regiao", "uf"],
        how="inner"
    ).drop(columns=["ano"])

    df_calculos["taxa_escolarizacao"] = (
        df_calculos.groupby("uf")["taxa_escolarizacao"]
        .transform(lambda x: x.ffill().bfill())
    )

    # --- Agrupamento das métricas SCR ---
    df_metricas = pd.concat([
        agrupar_metrica(df_calculos, "carteira_vencida").rename("carteira_vencida"),
        agrupar_metrica(df_calculos, "carteira_ativa").rename("carteira_ativa"),
        agrupar_metrica(df_calculos, "carteira_ativa", df_calculos["classe"].isin(["D", "E"])).rename("carteira_ativa_classes_de"),
        agrupar_metrica(df_calculos, "vencido_15_a_60_dias").rename("vencido_15_a_60_dias"),
        agrupar_metrica(df_calculos, "vencido_61_a_90_dias").rename("vencido_61_a_90_dias"),
        agrupar_metrica(df_calculos, "vencido_acima_de_90_dias").rename("vencido_acima_de_90_dias")
    ], axis=1).reset_index()

    df_calculos = (
        df_calculos[["ano_mes", "regiao", "uf", "taxa_escolarizacao"]]
        .drop_duplicates()
        .merge(df_metricas, on=filtro, how="left")
    )

    # --- Soma dos atrasos a partir de 15 dias ---
    df_calculos["total_vencido_15_mais"] = (
        df_calculos["vencido_15_a_60_dias"].fillna(0) +
        df_calculos["vencido_61_a_90_dias"].fillna(0) +
        df_calculos["vencido_acima_de_90_dias"].fillna(0)
    )

    # --- CORREÇÃO DA DIVISÃO POR ZERO (GJ00-32) ---
    # Uso de np.where para tratar denominadores nulos ou iguais a zero:
    df_calculos["inadimplenciaReal"] = np.where(
        df_calculos["carteira_ativa"] > 0,
        df_calculos["carteira_vencida"] / df_calculos["carteira_ativa"],
        0.0
    )

    df_calculos["fragilidadeRenda"] = np.where(
        df_calculos["carteira_ativa"] > 0,
        df_calculos["carteira_ativa_classes_de"] / df_calculos["carteira_ativa"],
        0.0
    )

    df_calculos["agingDivida"] = np.where(
        df_calculos["carteira_vencida"] > 0,
        df_calculos["total_vencido_15_mais"] / df_calculos["carteira_vencida"],
        0.0
    )

    df_calculos["vulnerabilidadeSocial"] = 1 - df_calculos["taxa_escolarizacao"]

    # --- Normalização ---
    df_calculos = normalizacao(df_calculos, "inadimplenciaReal")
    df_calculos = normalizacao(df_calculos, "fragilidadeRenda")
    df_calculos = normalizacao(df_calculos, "agingDivida")
    df_calculos = normalizacao(df_calculos, "vulnerabilidadeSocial")

    # --- Limpeza final de segurança ---
    df_calculos = df_calculos.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_calculos[["ano_mes", "regiao", "uf", "inadimplenciaReal", "fragilidadeRenda", "agingDivida", "vulnerabilidadeSocial"]]



# **Eixo II – Inclusão e Expansão Demográfica**

---



In [ ]:
# --- Mudança feita dia 24-09 (Miguel) ---

def calcular_eixo_ii(df_scr_pix_base, df_ibge_base):
    df_scr_pix = df_scr_pix_base.copy()
    df_ibge = df_ibge_base.copy()

    # --- Extrai ano e merge com IBGE ---
    df_scr_pix["ano"] = df_scr_pix["ano_mes"].astype(str).str[:4]

    df_calculos = df_scr_pix.merge(
        df_ibge[["ano", "regiao", "uf", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]],
        on=["ano", "regiao", "uf"],
        how="left"
    ).drop(columns=["ano"])

    def eliminar_duplicata(metrica):
        df_calculos[metrica] = (
            df_calculos.groupby("uf")[metrica]
            .transform(lambda x: x.ffill().bfill())
        )

    eliminar_duplicata("populacao_residente")
    eliminar_duplicata("taxa_crescimento")
    eliminar_duplicata("variacao_populacao")
    eliminar_duplicata("bonusDemografico")

    # --- Agrupamento das métricas Pix ---
    df_metricas = pd.concat([
        agrupar_metrica(df_calculos, "vl_pagador").rename("vl_pagador"),
        agrupar_metrica(df_calculos, "qt_pes_pagador").rename("qt_pes_pagador"),
        agrupar_metrica(df_calculos, "qt_pagador").rename("qt_pagador"),
    ], axis=1).reset_index()

    df_calculos = (
        df_calculos[["ano_mes", "regiao", "uf", "populacao_residente", "taxa_crescimento", "variacao_populacao", "bonusDemografico"]]
        .drop_duplicates()
        .merge(df_metricas, on=filtro, how="left")
    )

    # --- CORREÇÃO DO CÁLCULO E DIVISÃO POR ZERO (GJ00-33) ---
    # 1. Maturidade Uso PIX = qt_pagador / qt_pes_pagador
    maturidade_uso = np.where(
        df_calculos["qt_pes_pagador"] > 0,
        df_calculos["qt_pagador"] / df_calculos["qt_pes_pagador"],
        0.0
    )

    # 2. Maturidade Financeira PIX = vl_pagador / qt_pagador
    maturidade_financeira = np.where(
        df_calculos["qt_pagador"] > 0,
        df_calculos["vl_pagador"] / df_calculos["qt_pagador"],
        0.0
    )

    # 3. Consolidado Maturidade PIX = (Uso * 0.6) + (Financeira * 0.4)
    df_calculos["maturidadePix"] = (maturidade_uso * 0.6) + (maturidade_financeira * 0.4)

    # Demais indicadores do Eixo II
    df_calculos["crescimentoPopulacional"] = df_calculos["taxa_crescimento"]
    df_calculos["totalHabitantes"] = df_calculos["populacao_residente"]

    # --- Normalização ---
    df_calculos = normalizacao(df_calculos, "maturidadePix")
    df_calculos = normalizacao(df_calculos, "crescimentoPopulacional")
    df_calculos = normalizacao(df_calculos, "totalHabitantes")
    df_calculos = normalizacao(df_calculos, "bonusDemografico")

    # --- Limpeza final de segurança ---
    df_calculos = df_calculos.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_calculos[["ano_mes", "regiao", "uf", "maturidadePix", "crescimentoPopulacional", "totalHabitantes", "bonusDemografico"]]

---
#**EXECUÇÃO PRINCIPAL**
---

In [ ]:
# --- Extração e Pré-Processamento ---
processar()

construir_df_scr_pix()
construir_df_ibge()

df_scr_pix = extract_data('/content/dados/scr_pix.parquet')
df_ibge = extract_data('/content/dados/ibge.parquet')

df_eixo_i = calcular_eixo_i(df_scr_pix, df_ibge)
df_eixo_ii = calcular_eixo_ii(df_scr_pix, df_ibge)

df_ibge.drop(columns=["bonusDemografico"], inplace=True)

# --- Limpeza de segurança contra NaNs e infinitos antes do envio ---
df_eixo_i = df_eixo_i.replace([np.inf, -np.inf], np.nan).fillna(0)
df_eixo_ii = df_eixo_ii.replace([np.inf, -np.inf], np.nan).fillna(0)
df_scr_pix = df_scr_pix.replace([np.inf, -np.inf], np.nan).fillna(0)
df_ibge = df_ibge.replace([np.inf, -np.inf], np.nan).fillna(0)